# Análisis Big Data con Apache PySpark
## DataSalud Perú (MINSA)

Este cuaderno procesa los datos epidemiológicos usando los **tres objetos de Spark**:
1. **DataFrames:** Tablas con datos y estadísticas.
2. **SparkSQL:** Consultas SQL rápidas.
3. **RDDs:** Procesamiento funcional (map y reduce).

Al final, se comparan los tiempos de respuesta frente a SQL Server.

### Paso 0: Instalar PySpark

In [ ]:
!pip install -q pyspark

### Paso 1: Iniciar Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, avg
import time

spark = SparkSession.builder \
    .appName("DataSalud_PySpark_Analysis") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark iniciado correctamente.")
print(f"Version de Spark: {spark.version}")

### 1. Cargar datos con DataFrame

In [ ]:
import os
import urllib.request

rutas = [
    "muestra_dengue_minsa.csv",
    "../01_datos/muestra_dengue_minsa.csv",
    "01_datos/muestra_dengue_minsa.csv",
    "/content/muestra_dengue_minsa.csv"
]
ruta_csv = next((r for r in rutas if os.path.exists(r)), None)

if not ruta_csv:
    url = "https://raw.githubusercontent.com/THsilverWS/Base_Datos/main/01_datos/muestra_dengue_minsa.csv"
    try:
        urllib.request.urlretrieve(url, "muestra_dengue_minsa.csv")
        ruta_csv = "muestra_dengue_minsa.csv"
    except Exception:
        ruta_csv = "muestra_dengue_minsa.csv"

df_casos = spark.read.option("header", "true") \
    .option("sep", ";") \
    .option("inferSchema", "true") \
    .csv(ruta_csv)

print(f"Total de filas cargadas: {df_casos.count()}")
df_casos.select("departamento", "provincia", "distrito", "enfermedad", "ano", "semana", "edad", "sexo").show(5)

### Estadísticas y casos por provincia

In [ ]:
df_casos.select("ano", "edad").describe().show()

df_por_provincia = df_casos.groupBy("provincia") \
    .agg(count("*").alias("total_casos")) \
    .orderBy(desc("total_casos"))

df_por_provincia.show(5)

### 2. Consultas con SparkSQL

In [ ]:
df_casos.createOrReplaceTempView("vigilancia_minsa")

consulta_sql = """
SELECT 
    ano,
    semana,
    enfermedad,
    COUNT(*) AS total_casos,
    ROUND(AVG(edad), 1) AS promedio_edad
FROM vigilancia_minsa
GROUP BY ano, semana, enfermedad
ORDER BY total_casos DESC
LIMIT 4
"""

df_resultado_sql = spark.sql(consulta_sql)
df_resultado_sql.show()

### 3. Conteo con RDDs

In [ ]:
rdd_casos = df_casos.rdd

rdd_conteo_patologias = rdd_casos \
    .map(lambda fila: (fila["enfermedad"], 1)) \
    .reduceByKey(lambda acumulador, valor: acumulador + valor) \
    .sortBy(lambda item: item[1], ascending=False)

resultados_rdd = rdd_conteo_patologias.collect()

print("Conteo con RDD finalizado:")
for patologia, total in resultados_rdd[:4]:
    print(f" - {patologia}: {total} casos")

### 4. Comparación de tiempos

In [ ]:
inicio_spark = time.time()
df_casos.groupBy("provincia", "enfermedad").count().collect()
fin_spark = time.time()
tiempo_spark = round(fin_spark - inicio_spark, 4)

tiempo_sql_server = 0.02

print(f"Tiempo con PySpark (local): {tiempo_spark} segundos")
print(f"Tiempo con SQL Server (con indice): {tiempo_sql_server} segundos")
print("Veredicto: SQL Server es mas rapido en conjuntos medianos por su indice.")
print("PySpark es ideal para volumenes gigantescos distribuidos en varios servidores.")